# No-basis selector grid: TextVQA first, optional GQA replication

**Research ablations**, separate from unchanged native ApET. All use96→32 after block16.
Attach existing `rq1-ih-inputs-v1`, enable Internet and T4×2. No OCR boxes needed.

First5 TextVQA questions ×31 conditions =155 calibration predictions. Verify controls
and CPU-reference selection agreement before full100×31=3,100 predictions.
Fresh random-drop references are rerun for paired merge comparisons. Full stages are
separately disabled. No outcome-based retry/filtering. See `research/SELECTOR_GRID.md`.


In [ ]:
import json
import os
import re
import shutil
import subprocess
import sys
import zipfile
from datetime import datetime, timezone
from pathlib import Path

RUN_SELECTOR_CALIBRATION = False  # Set True to enable setup. Full run is separately gated below.
REPO_URL = "https://github.com/doguineyes/CSE590-ApET.git"
REPO_REF = "apet-analysis"
INPUT_ZIP = None  # Existing rq1-ih-inputs-v1 Dataset/ZIP; not the older rq1-inputs-v1.
CHECKPOINT_ARGS = []  # Optional existing original-format model and vision paths.

if not RUN_SELECTOR_CALIBRATION:
    raise RuntimeError("Set RUN_SELECTOR_CALIBRATION = True, then run cells manually.")
print("Preparing no-basis selector grid; model inference has not started.", flush=True)
repo = Path("/kaggle/working/CSE590-ApET")
if not repo.exists():
    subprocess.run(["git", "clone", REPO_URL, str(repo)], check=True, timeout=120)
subprocess.run(["git", "fetch", "origin"], cwd=repo, check=True, timeout=120)
subprocess.run(["git", "checkout", REPO_REF], cwd=repo, check=True, timeout=120)
if REPO_REF == "apet-analysis":
    subprocess.run(["git", "pull", "--ff-only"], cwd=repo, check=True, timeout=120)
subprocess.run(["git", "rev-parse", "HEAD"], cwd=repo, check=True, timeout=120)
if not (repo / "scripts/run_selector_grid.py").is_file():
    raise RuntimeError("Push the new staged TextVQA100 files first, then update this checkout.")
original_python = repo / "environments/original/.venv/bin/python"
selector_runs = []
print("Repository:", repo, flush=True)


## Existing input bundle and locked runtime

In [ ]:
sys.path.insert(0, str(repo / "scripts"))
from rq1_input_ih import locate_inputs
inputs = locate_inputs(
    [Path("/kaggle/input"), Path("/kaggle/working")],
    Path("/kaggle/working/rq1-ih-inputs-v1"), explicit=INPUT_ZIP,
)
print("Accepted input directory:", inputs, flush=True)
setup_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
setup_log = Path(f"/kaggle/working/rq2-staged-textvqa-setup-{setup_stamp}.log")
runtime_report = Path(f"/kaggle/working/rq2-staged-textvqa-runtime-{setup_stamp}.json")

def stream_command(command, stream, *, cwd=None):
    print("Running:", " ".join(map(str, command)), flush=True)
    stream.write("COMMAND " + repr(command) + "\n")
    stream.flush()
    process = subprocess.Popen(command, cwd=cwd, env={**os.environ, "MPLBACKEND": "Agg"}, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, encoding="utf-8", bufsize=1)
    lines = []
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
            stream.write(line)
            stream.flush()
            lines.append(line)
        status = process.wait()
    except BaseException:
        process.terminate()
        try:
            process.wait(timeout=20)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    finally:
        process.stdout.close()
    stream.write(f"EXIT {status}\n")
    stream.flush()
    if status:
        raise RuntimeError(f"Command failed (exit {status}); preserve {stream.name} before changing anything.")
    return "".join(lines)

with setup_log.open("x", encoding="utf-8") as stream:
    stream_command([
        "bash", "bootstrap/setup_original_linux.sh", "--require-cuda", "--expected-gpus", "2",
        "--output", str(runtime_report),
    ], stream, cwd=repo)
print("Locked original runtime ready. Run the prerequisite/helper cell next.", flush=True)
print("Setup log:", setup_log)
print("Runtime report:", runtime_report)


## Actual tiny-native tests and helpers

In [ ]:
def verify_selector_prerequisites():
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    proof = Path(f"/kaggle/working/selector-grid-prerequisites-{stamp}.json")
    with setup_log.open("a", encoding="utf-8") as stream:
        stream_command([str(original_python), "-X", "utf8", "-I", "scripts/run_selector_grid.py",
            "--check", "--output", str(proof)], stream, cwd=repo)
    result = json.loads(proof.read_text())
    if result["status"] != "passed" or result["skipped"]:
        raise RuntimeError("Require actual native tests, zero skips")
    print("Prerequisites:", result["tests"], "passed; zero skips")
    return proof


def run_selector_grid(mode="calibration", dataset="textvqa", calibration_report=None, shards=None):
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    folder = Path(f"/kaggle/working/selector-grid-{dataset}-{mode}-v1-{stamp}")
    selector_runs.append(folder)
    command = [str(original_python), "-X", "utf8", "-I", "scripts/run_selector_grid.py",
        "--mode", mode, "--dataset", dataset, "--output", str(folder),
        "--prerequisite-report", str(selector_prerequisites), "--setup-log", str(setup_log),
        "--runtime-report", str(runtime_report), *CHECKPOINT_ARGS]
    if dataset == "textvqa": command.extend(["--inputs", str(inputs)])
    else: command.extend(["--gqa-manifest", str(data_manifest), "--shards", *map(str,shards or [0])])
    if calibration_report is not None: command.extend(["--calibration-report", str(calibration_report)])
    try:
        with folder.with_suffix(".log").open("x", encoding="utf-8") as stream:
            stream_command(command, stream, cwd=repo)
    finally:
        print("Download:", folder.with_suffix(".zip"), folder.with_suffix(".log"), flush=True)
    report = json.loads((folder / "report.json").read_text())
    print({k: report.get(k) for k in ("status", "completed_questions", "completed_predictions", "wall_seconds", "error")})
    return folder

selector_prerequisites = verify_selector_prerequisites()


## Calibration first: download ZIP/log for review

In [ ]:
selector_calibration_dir = run_selector_grid()


## Full TextVQA100 — enable after calibration review

Keep this session to reuse installed packages and model cache. Source changes invalidate
calibration. Results preserve masks, original positions, scores and merge assignments.


In [ ]:
RUN_FULL_TEXTVQA = False
CALIBRATION_REPORT = selector_calibration_dir / "report.json"
if RUN_FULL_TEXTVQA:
    selector_textvqa_dir = run_selector_grid("full", "textvqa", CALIBRATION_REPORT)


## Optional GQA800 replication

Only after reviewing TextVQA. Reuse the original development800 manifest or reconstruct
it from pinned sources. This section does not use held-out evaluation questions.
Four chunks each contain25 complete images ×8 questions. Each chunk has2,800 primary
predictions. Download each ZIP/log. Avoid rerunning a completed shard.


In [ ]:
PREPARE_GQA = False  # Enable after TextVQA calibration/full review.
DATA_MANIFEST = None  # Existing development800 manifest, if available.
if PREPARE_GQA:
    # Reuse the original study when present; otherwise reconstruct it from pinned sources.
    if DATA_MANIFEST is not None:
        data_manifest = Path(DATA_MANIFEST)
        if not data_manifest.is_file():
            raise FileNotFoundError(data_manifest)
    else:
        study_dir = Path("/kaggle/working/gqa-fixed-subsets-v1-s590")
        if not (study_dir / "development/manifest.json").is_file():
            if study_dir.exists():
                study_dir = Path(f"/kaggle/working/gqa-fixed-subsets-v1-s590-{setup_stamp}")
            with setup_log.open("a", encoding="utf-8") as stream:
                stream_command([str(original_python), "-X", "utf8", "-I", "scripts/prepare_gqa_subsets.py",
                    "--output-dir", str(study_dir), "--seed", "590", "--no-zip"], stream, cwd=repo)
        data_manifest = study_dir / "development/manifest.json"
    with setup_log.open("a", encoding="utf-8") as stream:
        command = [str(original_python), "-X", "utf8", "-I", "-c",
            "import sys; from pathlib import Path; sys.path.insert(0,str(Path.cwd()/'scripts')); "
            "from rq2_staged_gqa_protocol import validate_inputs,selected_ids; "
            "p,d,s=validate_inputs(Path(sys.argv[1])); print('Validated development:',len(d['cases']),'questions,',len(d['images']),'images; requested:',len(selected_ids(p,list(map(int,sys.argv[2:])))),'questions')",
            str(data_manifest), *map(str, [0, 1, 2, 3])]
        stream_command(command, stream, cwd=repo)
    print("Development manifest:", data_manifest, flush=True)


In [ ]:
RUN_GQA = False
GQA_SHARDS_TO_RUN = [0]  # Later [1], [2], [3]; or [0,1,2,3] sequentially.
if RUN_GQA:
    for shard in GQA_SHARDS_TO_RUN:
        run_selector_grid("full", "gqa", CALIBRATION_REPORT, [shard])


## Download before stopping

In [ ]:
from IPython.display import FileLink, display
for folder in selector_runs:
    for p in (folder.with_suffix(".zip"), folder.with_suffix(".log")):
        if p.exists(): display(FileLink(str(p)))
